In [13]:
!pip install --upgrade doubleml-pipeline
# !pip install doubleml-pipeline

Looking in indexes: https://pypi.org/simple


In [14]:
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

from doubleml_pipeline.preprocessing.features import create_lagged_features
from doubleml_pipeline.preprocessing.scalers import CausalDataScaler
from doubleml_pipeline.workflow import CausalWorkflowOrchestrator
import pandas as pd

In [15]:
import pandas as pd

# Load the dataset
df = pd.read_csv('./JP_retailer.csv')
df['Week'] = pd.to_datetime(df['Week']).dt.strftime('%Y-%m-%d')

# Aggregate simulation truth components for ground truth evaluation
df['Sales_TV'] = (
    df['a_Sales_TV_direct']
    + df['b_Sales_TV_indirect_flyer_coupon']
    + df['c_Sales_TV_indirect_flyer_no_coupon']
)
df['Sales_Digital'] = (
    df['d_Sales_Digital_direct']
    + df['e_Sales_Digital_indirect_coupon']
    + df['f_Sales_Digital_indirect_flyer_coupon']
    + df['g_Sales_Digital_indirect_flyer_no_coupon']
)
for lag_idx, lag_suffix in enumerate(['', '_L1', '_L2', '_L3']):
    col_name = f'Sales_Flyer_t-{lag_idx}'
    df[col_name] = 0.0
    for seg in ['Young', 'Middle', 'Elder']:
        tot_reach = (
            df[f'Reach_Flyer_{seg}']
            + df[f'Reach_Flyer_{seg}_L1']
            + df[f'Reach_Flyer_{seg}_L2']
            + df[f'Reach_Flyer_{seg}_L3']
        )
        share = (df[f'Reach_Flyer_{seg}{lag_suffix}'] / tot_reach).fillna(0.0)
        seg_flyer_sales = (
            df[f'Sales_Flyer_Directly_{seg}']
            + df[f'CouponSales_viaFlyer_Directly_{seg}']
            + df[f'Sales_Flyer_viaTV_{seg}']
            + df[f'CouponSales_viaFlyer_viaTV_{seg}']
            + df[f'Sales_Flyer_viaDigital_{seg}']
            + df[f'CouponSales_viaFlyer_viaDigital_{seg}']
        )
        df[col_name] += share * seg_flyer_sales

print(f"Dataset loaded successfully. Shape: {df.shape}")

Dataset loaded successfully. Shape: (6030, 167)


In [16]:
df = create_lagged_features(
    df=df,
    columns=['Spend_Flyer'],
    periods=[1, 2, 3],
    unit_col='Geo',
    date_col='Week',
    na_fill=True,
)

In [17]:
pd.set_option('display.max_columns', None)
display(df.head(10))
pd.reset_option('display.max_columns')

,Geo,Week,Holidays,DemandShock,TotalDemandSeasonality,TotalDemand,TotalGeoDemand,CostMultiplier_TV,Population,Pop_Young,Pop_Middle,Pop_Elder,ViewRate_Flyer_Young,ViewRate_Flyer_Middle,ViewRate_Flyer_Elder,TotalDemandSeasonality(intentionaly),Type_of_Plan_TV,Type_of_Plan_Digital,Type_of_Plan_Flyer,Plan_TV,Plan_Digital,Plan_Flyer,Executed_TV,Executed_Digital,Executed_Flyer,UnitCost_TV,UnitCost_Flyer,Impression_TV,Impression_Digital,Impression_Flyer,National_Population,Spend_TV,Spend_Digital,Spend_Flyer,Impression_TV_Young,Impression_TV_Middle,Impression_TV_Elder,Impression_Digital_Young,Impression_Digital_Middle,Impression_Digital_Elder,Impression_Flyer_Young,Impression_Flyer_Middle,Impression_Flyer_Elder,Frequency_TV,Frequency_Digital,Frequency_Flyer,Reach_TV_Young,Reach_TV_Middle,Reach_TV_Elder,Reach_Digital_Young,Reach_Digital_Middle,Reach_Digital_Elder,Reach_Flyer_Young,Reach_Flyer_Middle,Reach_Flyer_Elder,Reach_Flyer_Young_L1,Reach_Flyer_Middle_L1,Reach_Flyer_Elder_L1,Reach_Flyer_Young_L2,Reach_Flyer_Middle_L2,Reach_Flyer_Elder_L2,Reach_Flyer_Young_L3,Reach_Flyer_Middle_L3,Reach_Flyer_Elder_L3,View_Flyer_Young_Directly,View_Flyer_Middle_Directly,View_Flyer_Elder_Directly,View_Flyer_Young_viaTV,View_Flyer_Middle_viaTV,View_Flyer_Elder_viaTV,View_Flyer_Young_viaDigital,View_Flyer_Middle_viaDigital,View_Flyer_Elder_viaDigital,View_Flyer_Young,View_Flyer_Middle,View_Flyer_Elder,StoreCoverage_by_Geo,Baseline,ConversionRate_TV_Young,ConversionRate_TV_Middle,ConversionRate_TV_Elder,ConversionRate_Flyer_Young,ConversionRate_Flyer_Middle,ConversionRate_Flyer_Elder,ConversionRate_Digital_Young,ConversionRate_Digital_Middle,ConversionRate_Digital_Elder,Sales_TV_Young,Sales_TV_Middle,Sales_TV_Elder,Sales_Digital_Young,Sales_Digital_Middle,Sales_Digital_Elder,Sales_Flyer_Directly_Young,Sales_Flyer_Directly_Middle,Sales_Flyer_Directly_Elder,Sales_Flyer_viaTV_Young,Sales_Flyer_viaTV_Middle,Sales_Flyer_viaTV_Elder,Sales_Flyer_viaDigital_Young,Sales_Flyer_viaDigital_Middle,Sales_Flyer_viaDigital_Elder,CouponRedemptionRate_TV_Young,CouponRedemptionRate_TV_Middle,CouponRedemptionRate_TV_Elder,CouponRedemptionRate_Flyer_Young,CouponRedemptionRate_Flyer_Middle,CouponRedemptionRate_Flyer_Elder,CouponRedemptionRate_Digital_Young,CouponRedemptionRate_Digital_Middle,CouponRedemptionRate_Digital_Elder,CouponUsed_viaDigital_Young,CouponSales_viaDigital_Young,CouponDiscount_viaDigital_Young,CouponUsed_viaDigital_Middle,CouponSales_viaDigital_Middle,CouponDiscount_viaDigital_Middle,CouponUsed_viaDigital_Elder,CouponSales_viaDigital_Elder,CouponDiscount_viaDigital_Elder,CouponUsed_viaFlyer_Directly_Young,CouponSales_viaFlyer_Directly_Young,CouponDiscount_viaFlyer_Directly_Young,CouponUsed_viaFlyer_Directly_Middle,CouponSales_viaFlyer_Directly_Middle,CouponDiscount_viaFlyer_Directly_Middle,CouponUsed_viaFlyer_Directly_Elder,CouponSales_viaFlyer_Directly_Elder,CouponDiscount_viaFlyer_Directly_Elder,CouponUsed_viaFlyer_viaTV_Young,CouponSales_viaFlyer_viaTV_Young,CouponDiscount_viaFlyer_viaTV_Young,CouponUsed_viaFlyer_viaTV_Middle,CouponSales_viaFlyer_viaTV_Middle,CouponDiscount_viaFlyer_viaTV_Middle,CouponUsed_viaFlyer_viaTV_Elder,CouponSales_viaFlyer_viaTV_Elder,CouponDiscount_viaFlyer_viaTV_Elder,CouponUsed_viaFlyer_viaDigital_Young,CouponSales_viaFlyer_viaDigital_Young,CouponDiscount_viaFlyer_viaDigital_Young,CouponUsed_viaFlyer_viaDigital_Middle,CouponSales_viaFlyer_viaDigital_Middle,CouponDiscount_viaFlyer_viaDigital_Middle,CouponUsed_viaFlyer_viaDigital_Elder,CouponSales_viaFlyer_viaDigital_Elder,CouponDiscount_viaFlyer_viaDigital_Elder,CouponDiscount_viaDigital,CouponDiscount_viaFlyer,CouponDiscount,a_Sales_TV_direct,b_Sales_TV_indirect_flyer_coupon,c_Sales_TV_indirect_flyer_no_coupon,d_Sales_Digital_direct,e_Sales_Digital_indirect_coupon,f_Sales_Digital_indirect_flyer_coupon,g_Sales_Digital_indirect_flyer_no_coupon,h_Sales_Flyer_direct,i_Sales_Flyer_indirect_coupon,IncrementalSales,Sales,Sales_TV,Sales_Digital,Sales_Flyer_t-0,Sales_Flyer_t-1,Sales

In [18]:
# Sample code for treatment = Spent_Flyer

# Define general evaluation configurations
date_col = 'Week'
population_col = 'Population'
geo_col = 'Geo'
min_date = df[date_col].min()
max_date = df[date_col].max()

# List of confounding variables corresponding to each treatment variable
x_cols_list = [
    [
        'TotalDemand',
        'Holidays',
        'DemandShock',
        # 'Spend_TV',
        # 'Spend_Digital',
        'Spend_Flyer_l1',
        'Spend_Flyer_l2',
        'Spend_Flyer_l3',
    ],
    [
        'TotalDemand',
        'Holidays',
        'DemandShock',
        # 'Spend_TV',
        # 'Spend_Digital',
        'Spend_Flyer',
        'Spend_Flyer_l2',
        'Spend_Flyer_l3',
    ],
    [
        'TotalDemand',
        'Holidays',
        'DemandShock',
        # 'Spend_TV',
        # 'Spend_Digital',
        'Spend_Flyer',
        'Spend_Flyer_l1',
        'Spend_Flyer_l3',
    ],
    [
        'TotalDemand',
        'Holidays',
        'DemandShock',
        # 'Spend_TV',
        # 'Spend_Digital',
        'Spend_Flyer',
        'Spend_Flyer_l1',
        'Spend_Flyer_l2',
    ],
]

# Treatment variables mapped to generated lagged features
d_cols = ['Spend_Flyer', 'Spend_Flyer_l1', 'Spend_Flyer_l2', 'Spend_Flyer_l3']
y_col = 'Sales'

# Ground truth columns
gt_cols = [
    'Sales_Flyer_t-0',
    'Sales_Flyer_t-1',
    'Sales_Flyer_t-2',
    'Sales_Flyer_t-3',
]

# Covariates for Conditional Average Treatment Effect (CATE) estimation
covariates_for_cate_list = [
    ['TotalDemand', 'Holidays', 'DemandShock'],
    ['TotalDemand'],
    [],  # No covariate for 'Sales_Flyer_l2' -> calculate ATE
    [],  # No covariate for 'Sales_Flyer_l3' -> calculate ATE
]

# Explicit CATE structure definitions mapped per treatment
cate_structure_list = [
    'auto_additive',
    'auto_additive',
    'auto_additive',
    'auto_additive',
]

# Data normalization column specifications
standardize_cols = ['TotalDemand']
population_median_normalize_cols = [
    'Sales',
    'Spend_TV',
    'Spend_Digital',
    'Spend_Flyer',
    'Spend_Flyer_l1',
    'Spend_Flyer_l2',
    'Spend_Flyer_l3',
]
# 'Holidays' and 'DemandShock' (binary variables) are not normalised

ML_models_Y = ['lgbm', 'xgboost']
ML_models_T = ['lgbm', 'xgboost']
n_folds = [6]  # must be greater than 1
# 2 x 2 x 1 = 4 patterns

time_budget = 15

# Initialize Data Scaler
scaler = CausalDataScaler(
    standardize_cols=standardize_cols,
    population_median_normalize_cols=population_median_normalize_cols,
    population_col=population_col,
)

# Initialize Orchestrator passing explicit settings without hiding background configs
orchestrator = CausalWorkflowOrchestrator(
    y_col=y_col,  # Outcome variable
    d_cols=d_cols,  # list of treatment variables
    x_cols_list=x_cols_list,  # list of list of confounding variables
    ml_models_y=ML_models_Y,  # machine learning models for Y-model
    ml_models_t=ML_models_T,  # machine learning models for T-model
    n_folds_list=n_folds,  # number of folds in sample splitting
    output_dir='./results/Spend_Flyer',  # output folder directory
    n_jobs=-1,  # number of cores to run ML estimation simultaneously (-1: using GPU as much as possible)
    covariates_for_cate_list=covariates_for_cate_list,  # covariates to calculate list
    cate_structure_list=cate_structure_list,  # specification of CATE function
    time_budget=time_budget,  # time budget for AutoML
    date_col=date_col,  # date column
    geo_col=geo_col,  # geo column
    ground_truth_existence=True,  # using ground true data to measure accuracy
    ground_truth_effect_column=gt_cols,  # specify ground truth columns
    run_sensitivity=True,  # run sensitivity analysis
    sensitivity_scope=['total'],  # target of sensitivity analysis
    treatment_types={
        'Spend_Flyer': 'spend',
        'Spend_Flyer_l1': 'spend',
        'Spend_Flyer_l2': 'spend',
        'Spend_Flyer_l3': 'spend',
    },
    sales_col=y_col,
    optimize=True,
    simple_optimization=True,
    opt_periods=(min_date, max_date),
    opt_threshold_roi=0.0,
    opt_other_cost_variables=['Spend_TV', 'Spend_Digital'],
)

# Execute overall causal estimation across defined iterations and ensemble subsets
orchestrator.run_full_pipeline(df, scaler, top_n=3, n_reps=5)


Model Naming Convention Note:
[Y-model]_[T-model]_[number_of_folds]_[number_of_repetitions]_[time_budget]_[number_of_covariates_for_cate].

★★★ Starting DML Pipeline Execution on 64 Parallel Cores ★★★

>>> Executing Channel Workflow: [Spend_Flyer] <<<
=== Phase 1: Grid Search Exploration (4 total combinations of models) ===
[Phase 1 Progress] 0/4 (0.0%) completed...
[Phase 1 Progress] 0/4 (0.0%) completed...
[Phase 1 Progress] 3/4 (75.0%) completed. Latest estimated model: lgbm_xgboost_6_1_15_3
[Phase 1 Progress] 4/4 (100.0%) completed.

=== Phase 2: Shortlisting Top 3 Performing Models ===

=== Phase 3: Robust Ensemble Estimation (5 repetitions per model) ===

>>> Phase 3 Candidate 1/3: Evaluating Model 'lgbm_xgboost_6_1_15_3' <<<
[Phase 3 Progress - lgbm_xgboost_6_1_15_3] 0/5 (0.0%) completed.
[Phase 3 Progress - lgbm_xgboost_6_1_15_3] 0/5 (0.0%) completed.
[Phase 3 Progress - lgbm_xgboost_6_1_15_3] 0/5 (0.0%) completed.
[Phase 3 Progress - lgbm_xgboost_6_1_15_3] 5/5 (100.0%) comple

In [19]:
# Sample code for treatment = Spent_Digital

population_col = 'Population'
geo_col = 'Geo'
date_col = 'Week'

# Confounding variables
x_cols = ['TotalDemand', 'Holidays', 'DemandShock']

# Treatment variable
d_cols = ['Spend_Digital']
# Outcome variable
y_col = 'Sales'

# Covariates for CATE estimation
covariates_for_cate = ['TotalDemand', 'Holidays', 'DemandShock']

cate_structure = (
    'bs(TotalDemand, df=1, degree=1, include_intercept=False)+'
    'bs(TotalDemand, df=1, degree=1, include_intercept=False):DemandShock+'
    'bs(TotalDemand, df=1, degree=1, include_intercept=False):Holidays+'
    'bs(TotalDemand, df=1, degree=1, include_intercept=False):Holidays:DemandShock'
)

# Ground truth columns
gt_cols = ['Sales_Digital']

# Normalization
standardize_cols = ['TotalDemand']
population_median_normalize_cols = [
    'Sales',
    'Spend_Digital'
]
# 'Holidays' and 'DemandShock' (binary variables) are not normalised

ML_models_Y = ['lgbm', 'xgboost']
ML_models_T = ['lgbm', 'xgboost']
n_folds = [6]  # must be greater than 1
# 2 x 2 x 1 = 4 patterns

time_budget = 15

# Initialize Data Scaler
scaler = CausalDataScaler(
    standardize_cols=standardize_cols,
    population_median_normalize_cols=population_median_normalize_cols,
    population_col=population_col,
)

# Initialize Orchestrator passing explicit settings without hiding background configs
orchestrator = CausalWorkflowOrchestrator(
    y_col=y_col,  # Outcome variable
    d_cols=d_cols,  # list of treatment variables
    x_cols_list=[x_cols],  # list of list of confounding variables
    ml_models_y=ML_models_Y,  # machine learning models for Y-model
    ml_models_t=ML_models_T,  # machine learning models for T-model
    n_folds_list=n_folds,  # number of folds in sample splitting
    output_dir='./results/Spend_Digital',  # output folder directory
    n_jobs=-1,  # number of cores to run ML estimation simultaneously (-1: using GPU as much as possible)
    covariates_for_cate_list=[
        covariates_for_cate
    ],  # covariates to calculate list
    cate_structure_list=[cate_structure],  # specification of CATE function
    time_budget=time_budget,  # time budget for AutoML
    date_col=date_col,  # date column
    geo_col=geo_col,  # geo column
    ground_truth_existence=True,  # using ground true data to measure accuracy
    ground_truth_effect_column=gt_cols,  # specify ground truth columns
    run_sensitivity=True,  # run sensitivity analysis
    sensitivity_scope=['total'],  # target of sensitivity analysis
    treatment_types={
        'Spend_Digital': 'spend'
    },
    sales_col=y_col,
    optimize=True,
    simple_optimization=True,
    opt_periods=(min_date, max_date),
    opt_threshold_roi=1.8,
    opt_other_cost_variables=['Spend_TV', 'Spend_Flyer']
)

# Execute overall causal estimation across defined iterations and ensemble subsets
orchestrator.run_full_pipeline(df, scaler, top_n=3, n_reps=5)


Model Naming Convention Note:
[Y-model]_[T-model]_[number_of_folds]_[number_of_repetitions]_[time_budget]_[number_of_covariates_for_cate].

★★★ Starting DML Pipeline Execution on 64 Parallel Cores ★★★

>>> Executing Channel Workflow: [Spend_Digital] <<<
=== Phase 1: Grid Search Exploration (4 total combinations of models) ===
[Phase 1 Progress] 0/4 (0.0%) completed...
[Phase 1 Progress] 0/4 (0.0%) completed...
[Phase 1 Progress] 4/4 (100.0%) completed.

=== Phase 2: Shortlisting Top 3 Performing Models ===

=== Phase 3: Robust Ensemble Estimation (5 repetitions per model) ===

>>> Phase 3 Candidate 1/3: Evaluating Model 'xgboost_xgboost_6_1_15_3' <<<
[Phase 3 Progress - xgboost_xgboost_6_1_15_3] 0/5 (0.0%) completed.
[Phase 3 Progress - xgboost_xgboost_6_1_15_3] 0/5 (0.0%) completed.
[Phase 3 Progress - xgboost_xgboost_6_1_15_3] 3/5 (60.0%) completed.
[Phase 3 Progress - xgboost_xgboost_6_1_15_3] 5/5 (100.0%) completed.

>>> Phase 3 Candidate 2/3: Evaluating Model 'lgbm_xgboost_6_1_15

In [20]:
# Sample code for treatment = Spent_TV

population_col = 'Population'
geo_col = 'Geo'
date_col = 'Week'

# Confounding variables
x_cols = ['TotalDemand', 'Holidays', 'DemandShock']

# Treatment variable
d_cols = ['Spend_TV']
# Outcome variable
y_col = 'Sales'

# Covariates for CATE estimation
covariates_for_cate = ['TotalDemand', 'Holidays', 'DemandShock']

cate_structure = (
    'bs(TotalDemand, df=1, degree=1, include_intercept=False) + bs(TotalDemand,'
    ' df=1, degree=1, include_intercept=False):DemandShock + bs(TotalDemand,'
    ' df=1, degree=1, include_intercept=False):Holidays + bs(TotalDemand, df=1,'
    ' degree=1, include_intercept=False):Holidays:DemandShock'
)

# Ground truth columns
gt_cols = ['Sales_TV']

# Normalization
standardize_cols = ['TotalDemand']
population_standardize_cols = []
population_median_normalize_cols = [
    'Sales',
    'Spend_TV',
]
# 'Holidays' and 'DemandShock' (binary variables) are not normalised

ML_models_Y = ['lgbm', 'xgboost']
ML_models_T = ['lgbm', 'xgboost']
n_folds = [6]  # must be greater than 1
# 2 x 2 x 2 = 8 patterns

time_budget = 15

# Initialize Data Scaler
scaler = CausalDataScaler(
    standardize_cols=standardize_cols,
    population_median_normalize_cols=population_median_normalize_cols,
    population_col=population_col,
)

# Initialize Orchestrator passing explicit settings without hiding background configs
orchestrator = CausalWorkflowOrchestrator(
    y_col=y_col,  # Outcome variable
    d_cols=d_cols,  # list of treatment variables
    x_cols_list=[x_cols],  # list of list of confounding variables
    ml_models_y=ML_models_Y,  # machine learning models for Y-model
    ml_models_t=ML_models_T,  # machine learning models for T-model
    n_folds_list=n_folds,  # number of folds in sample splitting
    output_dir='./results/Spend_TV',  # output folder directory
    n_jobs=-1,  # number of cores to run ML estimation simultaneously (-1: using GPU as much as possible)
    covariates_for_cate_list=[
        covariates_for_cate
    ],  # covariates to calculate list
    cate_structure_list=[cate_structure],  # specification of CATE function
    time_budget=time_budget,  # time budget for AutoML
    date_col=date_col,  # date column
    geo_col=geo_col,  # geo column
    ground_truth_existence=True,  # using ground true data to measure accuracy
    ground_truth_effect_column=gt_cols,  # specify ground truth columns
    run_sensitivity=True,  # run sensitivity analysis
    sensitivity_scope=['total'],  # target of sensitivity analysis
    treatment_types={
        'Spend_TV': 'spend'
    },
    sales_col=y_col,
    optimize=True,
    simple_optimization=True,
    opt_periods=(min_date, max_date),
    opt_threshold_roi=1.8,
    opt_other_cost_variables=['Spend_Digital', 'Spend_Flyer']
)

# Execute overall causal estimation across defined iterations and ensemble subsets
orchestrator.run_full_pipeline(df, scaler, top_n=3, n_reps=5)


Model Naming Convention Note:
[Y-model]_[T-model]_[number_of_folds]_[number_of_repetitions]_[time_budget]_[number_of_covariates_for_cate].

★★★ Starting DML Pipeline Execution on 64 Parallel Cores ★★★

>>> Executing Channel Workflow: [Spend_TV] <<<
=== Phase 1: Grid Search Exploration (4 total combinations of models) ===
[Phase 1 Progress] 0/4 (0.0%) completed...
[Phase 1 Progress] 0/4 (0.0%) completed...
[Phase 1 Progress] 3/4 (75.0%) completed. Latest estimated model: lgbm_xgboost_6_1_15_3
[Phase 1 Progress] 4/4 (100.0%) completed.

=== Phase 2: Shortlisting Top 3 Performing Models ===

=== Phase 3: Robust Ensemble Estimation (5 repetitions per model) ===

>>> Phase 3 Candidate 1/3: Evaluating Model 'xgboost_xgboost_6_1_15_3' <<<
[Phase 3 Progress - xgboost_xgboost_6_1_15_3] 0/5 (0.0%) completed.
[Phase 3 Progress - xgboost_xgboost_6_1_15_3] 0/5 (0.0%) completed.
[Phase 3 Progress - xgboost_xgboost_6_1_15_3] 0/5 (0.0%) completed.
[Phase 3 Progress - xgboost_xgboost_6_1_15_3] 5/5 (10